# Modelforge OpenMM Wrapper

## Simple Example

This notebook demonstrates the steps required to use a potential trained using modelforge as a `PythonFroce` within OpenMM.

### Load modelforge potential file
First let us import some various modules from modelforge we require for reading in a checkpoint file from a trained model:

In [ ]:
from modelforge.potential.potential import load_inference_model_from_checkpoint

# helper functions to load in the example model checkpoint file included in the repository
from modelforge.utils.io import get_path_string
from modelforge.ase.tests import data

# checkpoint file is saved in tests/data
checkpoint_file_path = get_path_string(data) + "/model.ckpt"
potential = load_inference_model_from_checkpoint(checkpoint_file_path, jit=False)


### Generate OpenMM topology
Next, we will create an OpenMM topology object.  This uses a simple helper function that generates a molecule using RDKit. 

In [ ]:
from modelforge.openmm.examples.helper_functions import openmm_topology_from_smiles

topology, positions = openmm_topology_from_smiles(smiles='NCCCCCO', optimize=True)
atomic_numbers = [atom.element.atomic_number for atom in topology.atoms()]


### Define OpenMM PythonForce

To allow us to interface with OpenMM, we wrap the modelforge potential, using the `generate_compute` function, and then pass this to OpenMM's PythonForce. 

In [ ]:
from modelforge.openmm.potential import generate_compute
from openmm import PythonForce

comp = generate_compute(potential=potential, atomic_numbers=atomic_numbers)
system_force = PythonForce(comp)

### OpenMM simulation setup

We then need to define a standard simulation workflow for OpenMM.

In [ ]:
import openmm
from openmm.unit import (
    kelvin,
    picosecond,
    femtosecond,
    nanometer,
    kilojoules_per_mole,
)
from openmm.app import PDBFile

# define the systme
system = openmm.System()
for atom in topology.atoms():
    system.addParticle(atom.element.mass)

# add the system_force instance defined above that wraps modelforge potential for PythonFroce
system.addForce(system_force)


import sys
from openmm import LangevinMiddleIntegrator
from openmm.app import Simulation, StateDataReporter, DCDReporter

# Create an integrator with a time step of 1 fs
temperature = 298.15 * kelvin
frictionCoeff = 1.0 / femtosecond
timeStep = 1.0 * femtosecond
integrator = LangevinMiddleIntegrator(temperature, frictionCoeff, timeStep)

# Create a simulation and set the initial positions and velocities
simulation = Simulation(topology, system, integrator)
simulation.context.setPositions(positions)

reporter = StateDataReporter(
    file=sys.stdout,
    reportInterval=100,
    step=True,
    time=True,
    potentialEnergy=True,
    temperature=True,
)
simulation.reporters.append(reporter)

# write out a trajectory and pdb file so we can visualize
simulation.reporters.append(DCDReporter('trajectory.dcd', 100))

state = simulation.context.getState(getPositions=True, getEnergy=True)
print("initial_energy ", state.getPotentialEnergy())


with open('initial_config.pdb', 'w') as f:
    PDBFile.writeFile(simulation.topology, state.getPositions(), f)
simulation.step(10000)